### Dataset and Task Metadata

In [1]:
from tokenize import group

from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sepsis_prediction_1m",
    version_from_unique_name="sepsis_prediction",
    version_comment="""
We sub-sample to 1 million train rows after splitting and test 250k rows.
""",
    # same as sepsis_prediction.ipynb
    dataset_year="2019",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/salikhussaini49/prediction-of-sepsis", # alt https://physionet.org/content/challenge-2019/1.0.0/
    download_description="""
We download the data from Kaggle as the link for the data from the original PhysioNet challenge seems to be down.

kaggle datasets download -d salikhussaini49/prediction-of-sepsis && unzip prediction-of-sepsis.zip all_files && rm prediction-of-sepsis.zip
mkdir -p local-data-warehouse/sepsis_prediction && mv all_files local-data-warehouse/sepsis_prediction/
""",
    # References
    academic_reference_bibtex="""@article{reyna2020early,
  title={Early prediction of sepsis from clinical data: the PhysioNet/Computing in Cardiology Challenge 2019},
  author={Reyna, Matthew A and Josef, Christopher S and Jeter, Russell and Shashikumar, Supreeth P and Westover, M Brandon and Nemati, Shamim and Clifford, Gari D and Sharma, Ashish},
  journal={Critical care medicine},
  volume={48},
  number={2},
  pages={210--217},
  year={2020},
  publisher={LWW}
}
""",
    academic_reference_bibtex_key="reyna2020early",
    license="ODC Open Database License", # CC BY-NC-SA 4.0 on Kaggle...
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We start with all files from Kaggle.

The original data has one file per user that was already preprocessed to one .csv file by the competition creators. Here we start with the preprocessed single .csv file. The data is non-IID in nature based on the groups of patients from different hospitals. The data that is grouped per patient ("Patient_ID"). These groups are also temporal in nature, but this temporal dependency is irrelevant as teh task is to predicts for one full patient (i.e., no refitting given patient information).
In the original competition, one had to predict for unseen patients from the existing hospitals and also for a new hidden hospital. In the public data, we only have two hospitals, (A) and (B). Given the limited data, we decide not to simulate a domain shift as we could not "train" for domain shift. Thus, we simulate only a normal grouped non-IID scenario. That is, we use all patients from both hospitals for training and testing, but ensure that the splits are grouped by patient ID. Thus, we simulate what would happen if someone trains a model on data from two hospitals and uses this to predict for other patients from these hospitals. This decision is also amplified by the large gap in performance for the unseen hospital in the competition (Report, Table 3), clearly pointing to a domain shift that is out-of-scope for this task. Note, we do not have temporal information of the order of patients, thus, we cannot simulate to only predict for patients "from the future". We believe this does not introduce any data leakage for this dataset.

- Patients with an ID larger than 100_000 are from hospital (B), while patients with an ID smaller than 100_000 are from hospital (A). We add this indicator into our data and then reset the Patient_IDs to be continuous increasing integers.
- The data consists of a lot of missing values due to missing measurements.
- Note, hour is a reset time index per patient. ICULOS is similar, but with an offset. We keep both as this can show that we have truncated data for a patient.
- We reverse the ordinal encoding of Gender.
- We mark features as categorical where appropriate.
- For each patient, we predict the sepsis status over time. So it is more or less a transformed survival task (also in the original challenge).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="SepsisLabel",
    problem_type="binary_classification",
    objective_metric_name="PhysioNet2019UtilityFunction", # https://github.com/physionetchallenges/evaluation-2019/blob/master/evaluate_sepsis_score.py
    stratify_on="SepsisLabel",
    group_on="Patient_ID",
    group_time_on="Hour",
    group_labels="per_sample",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "all_files" / "Dataset.csv")
print("Loaded data shape:", df.shape)

# Add Hospital Indicator
df["Hospital"] = "Hospital_A"
df.loc[df["Patient_ID"] > 100_000, "Hospital"] = "Hospital_B"
# Reset Patient IDs to be continuous (after remapping, IDs higher than 20k are from Hospital B)
codes, _ = pd.factorize(df["Patient_ID"])
df["Patient_ID"] = codes

df["Gender"] = df["Gender"].replace({0: "Female", 1: "Male"})

as_cat_type = ["Hospital", "SepsisLabel", "Patient_ID", "Gender", "Unit1", "Unit2"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.drop(columns=["Unnamed: 0"])

# Remove order of patients so that method figure this our themselves (remove order by "Patient_ID", "Hour")
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

Loaded data shape: (1552210, 44)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
    duplicate_column_check=False, # We know the data has unique columns
)


#### Dataset Overview
Rows: 1,552,210
Columns: 44
Use sampling: True (sample size: 155,221)
Get missing and unique counts per column...


missing/unique per-col:   0%|          | 0/44 [00:00<?, ?it/s]

Get example values per column...


examples per-col:   0%|          | 0/44 [00:00<?, ?it/s]

Get numeric feature statistics...


...

Get cat stats...


cat stats:   0%|          | 0/6 [00:00<?, ?it/s]

Get target stats...
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Patient_ID', 'HospAdmTime', 'Age', 'TroponinI', 'AST', 'PTT', 'Creatinine', 'Lactate', 'Glucose', 'SBP']


Rows remaining as candidates after top-10 filter: 662,154 (of 1,552,210)



#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,FiO2,pH,PaCO2,SaO2,AST,BUN,Alkalinephos,Calcium,Chloride,Creatinine,Bilirubin_direct,Glucose,Lactate,Magnesium,Phosphate,Potassium,Bilirubin_total,TroponinI,Hct,Hgb,PTT,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID,Hospital
0,14,68.0,98.0,36.80,124.0,88.00,66.0,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.00,Male,NaN,NaN,-104.13,15,0,37206,Hospital_B
1,48,98.0,100.0,NaN,151.0,111.00,86.0,16.0,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.00,Female,NaN,NaN,-2.80,49,0,24182,Hospital_B
2,4,75.0,100.0,35.39,88.0,56.00,NaN,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82.33,Male,NaN,NaN,-2.66,6,0,4273,Hospital_A
3,7,95.0,97.0,NaN,118.0,80.67,NaN,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.65,Female,1.0,0.0,-0.02,8,0,17236,Hospital_A
4,11,46.0,99.0,NaN,169.0,111.00,77.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69.00,Male,1.0,0.0,-5.06,12,0,24773,Hospital_B


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Unit1,category,611960.0,39.43,2.0,"0.0, 1.0"
1,Unit2,category,611960.0,39.43,2.0,"1.0, 0.0"
2,Gender,category,0.0,0.00,2.0,"Male, Female"
3,SepsisLabel,category,0.0,0.00,2.0,"0, 1"
4,Patient_ID,category,0.0,0.00,40336.0,"23156, 37860, 5150, 19786, 36144, 2194, 35884, 38992, 28921, 22913"
5,Hospital,category,0.0,0.00,2.0,"Hospital_A, Hospital_B"
6,Bilirubin_direct,float64,1549220.0,99.81,280.0,"0.1, 0.2, 0.3, 0.4, 0.7, 0.8, 1.6, 0.5, 0.6, 1.2"
7,Fibrinogen,float64,1541968.0,99.34,823.0,"153.0, 280.0, 332.0, 218.0, 162.0, 225.0, 281.0, 180.0, 185.0, 284.0"
8,TroponinI,float64,1537429.0,99.05,2423.0,"0.01, 0.03, 0.02, 0.04, 0.05, 0.06, 40.0, 0.07, 0.1, 0.08"
9,Bilirubin_total,float64,1529069.0,98.51,407.0,"0.6, 0.5, 0.4, 0.7, 0.8, 0.3, 0.9, 1.0, 1.1, 0.2"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Hour,155221.0,25.557412,28.933939,0.00,334.00
HR,139780.0,84.638155,17.311593,22.00,192.00
O2Sat,134865.0,97.194621,2.974821,20.00,100.00
Temp,52370.0,36.977395,0.767408,23.60,41.80
SBP,132578.0,123.691717,23.243067,27.00,298.00
MAP,135816.0,82.329735,16.312823,22.00,298.00
DBP,106697.0,63.781721,13.951569,20.00,298.00
Resp,131467.0,18.723116,5.079665,1.00,99.00
EtCO2,5743.0,33.109612,7.903604,10.00,100.00
BaseExcess,8570.0,-0.666645,4.405702,-28.00,24.00


In [7]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column      rank                           
Gender      1           Male   86741  55.88
            2         Female   68480  44.12
Hospital    1     Hospital_A   79075  50.94
            2     Hospital_B   76146  49.06
Patient_ID  1          23156      51   0.03
            2          37860      40   0.03
            3           5150      38   0.02
            4          19786      37   0.02
            5          36144      37   0.02
SepsisLabel 1              0  152386  98.17
            2              1    2835   1.83
Unit1       1           <NA>   61504  39.62
            2            0.0   47093  30.34
            3            1.0   46624  30.04
Unit2       1           <NA>   61504  39.62
            2            1.0   47093  30.34
            3            0.0   46624  30.04

In [8]:
# Target Distribution
target_df

,count,pct
SepsisLabel,,
0,1524294,98.2
1,27916,1.8


## Task Curation

In [9]:
# We create 1M train and 250k test rows
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=1,
    n_splits=1,
    group_on=task_mold.group_on,
    test_size=250_000,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)
train_index, test_index = splits[0][0]

# Resample train set to 1M rows
from sklearn.model_selection import StratifiedGroupKFold

df_train = df.iloc[train_index].reset_index(drop=True)
group_col = task_mold.group_on
n_groups = df_train[group_col].nunique()

test_size = 1_000_000
frac_of_samples_needed = test_size / len(train_index)
target_n_groups = round((1 - frac_of_samples_needed) * n_groups)
approximate_splits = round(n_groups / target_n_groups)
print(target_n_groups, approximate_splits)

splitter_inst = StratifiedGroupKFold(n_splits=approximate_splits, shuffle=True, random_state=43)
new_train_index, _ = next(splitter_inst.split(X=df_train, y=df_train[task_mold.stratify_on], groups=df_train[group_col]))

# Rest df and splits
df = pd.concat([df.iloc[train_index].iloc[new_train_index], df.iloc[test_index]], axis=0).reset_index(drop=True)
train_index = list(range(len(new_train_index)))
test_index = list(range(len(new_train_index), len(df)))

splits = {0: {0: (train_index, test_index)}}

print(f"Final train size: {len(train_index)}, Final test size: {len(test_index)}")
print(f"Final train groups: {df.iloc[train_index][group_col].nunique()}, Final test groups: {df.iloc[test_index][group_col].nunique()}")
print(f"\tFinal train target distribution: {df.iloc[train_index][task_mold.target_column_name].value_counts(normalize=True)}")
print(f"\tFinal test target distribution: {df.iloc[test_index][task_mold.target_column_name].value_counts(normalize=True)}")

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create a stratified grouped holdout split with 250k rows.",
    splits=splits
)

Using Stratified Grouped splits.
Using label-per-sample grouped splits.


Repeat 0, Fold 0:
            Train N: 1293699, Test N: 258511
            Target Distribution:
            	Train target distribution: {0: 0.9820275040793879, 1: 0.017972495920612137}
            	Test target distribution: {0: 0.9819543462367172, 1: 0.0180456537632828}
            Group Distribution Patient_ID:
            	Train: 33615
            	Test: 6721
            


7631 4


Final train size: 970175, Final test size: 258511


Final train groups: 25209, Final test groups: 6721
	Final train target distribution: SepsisLabel
0    0.982266
1    0.017734
Name: proportion, dtype: float64


	Final test target distribution: SepsisLabel
0    0.981954
1    0.018046
Name: proportion, dtype: float64


## Export

In [10]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to sepsis_prediction/versions/019d7391-f36e-72f8-89d7-f7ca71725034


019d7391-f36e-72f8-89d7-f7ca71725034
a7a7262a9a03806108c5639d92b7d60ffc3336734a90bdd990013207aac51f7e
